In [35]:
from skyfield.api import load, wgs84
import numpy as np
from numpy import array2string
from datetime import datetime, timedelta
import os
from scipy import io as sio
import pytz

In [40]:
# TODO: define constants and set other parameters
c = 299792458.0 # speed of light in m/s
fc = 11.325e9 # carrier frequency in Hz
lo = 9.75e9 # local oscillator frequency in Hz
fs = 1e6 # sampling frequency in Hz

In [46]:
# Load tle from path
path = '/Users/ishani/UIUC/Research/starlink_ofdm/AoA_Sim/tles/10_31/STARLINK-6150.tle'
satellites = load.tle_file(path)
print(satellites)
# initialize ground station with lat, lon = +40.11389690596162, -88.22484814695467
gs = wgs84.latlon(+40.11389690596162, -88.22484814695467)
# set start time and stop time, add 5 hours to account for time zone difference
ts = load.timescale()
start_time = datetime(2023, 10, 31, 0, 0, 0, tzinfo=pytz.UTC)+timedelta(hours=5) 
stop_time = datetime(2023, 10, 31, 0, 1, 0, tzinfo=pytz.UTC)+timedelta(hours=5)
# get start time as ts.utc
t = ts.utc(start_time)
n = np.arange(0, fs, 1)
i = 0
while True:
    dist = (satellites[0]-gs).at(t)
    el, az, distance = dist.altaz()
    # print time, elevation, azimuth, latitude, longitude
    print(t.utc_iso(), el.degrees, az.degrees)
    _, _, the_range, _, _, range_rate = dist.frame_latlon_and_rates(gs)
    doppler_shift = fc*range_rate.km_per_s*1000/c
    
    # generate received signal TODO: double check
    received_sig_0 = np.exp(-1j*(2*np.pi*fc+doppler_shift)*(n[i]/fs+the_range.km/c)) # TODO: can comment this out
    # downocnvert to baseband
    received_sig_0 = received_sig_0*np.exp(-1j*(2*np.pi*doppler_shift)*(n[i]/fs+the_range.km/c))

    # TODO: replicate code for more antennas

    # TODO: get phase difference between antennas

    #TODO: get AoA

    # increment t which is ts.utc by 1 second
    t = ts.utc(t.utc_datetime()+timedelta(seconds=1))
    i += 1

    if t.utc_datetime() > stop_time:
        break


[<EarthSatellite STARLINK-6150 catalog #57119 epoch 2023-10-31 06:43:42 UTC>]
2023-10-31T05:00:00Z 75.62354131239901 288.53448337241747
2023-10-31T05:00:01Z 76.32035236580272 288.5505039958126
2023-10-31T05:00:02Z 77.02080491394673 288.56800719191034
2023-10-31T05:00:03Z 77.72474384977549 288.5872558095727
2023-10-31T05:00:04Z 78.43200855676086 288.6085787002862
2023-10-31T05:00:05Z 79.14243306593973 288.63239282269717
2023-10-31T05:00:06Z 79.85584622726977 288.6592348678095
2023-10-31T05:00:07Z 80.57207189235785 288.689807559815
2023-10-31T05:00:08Z 81.29092910308012 288.7250492425461
2023-10-31T05:00:09Z 82.0122322775869 288.7662416650973
2023-10-31T05:00:10Z 82.73579137745492 288.81518289223266
2023-10-31T05:00:11Z 83.46141202685794 288.8744763870554
2023-10-31T05:00:12Z 84.18889552519332 288.9480388033293
2023-10-31T05:00:13Z 84.91803862787087 289.0420473769966
2023-10-31T05:00:14Z 85.64863280279312 289.1668457321812
2023-10-31T05:00:15Z 86.3804621968562 289.3411685171737
2023-10-3